# Data Platform Exploration

## Three-Database Architecture
- **PostgreSQL** (Relational — OLTP, 3NF)
- **MongoDB** (Document — Embedding & Referencing)
- **Neo4j** (Graph — Recommendations)

## 1. Setup & Imports

In [ ]:
import asyncio
import sys
sys.path.insert(0, '..')

from src.app.core.config.settings import get_settings
from src.app.db.relational.postgres_service import (
    get_connection, execute_ddl, seed_all,
    fetch_customers_with_purchases, fetch_top_products,
    fetch_product_recommendations
)
from src.app.db.nosql.mongo_service import (
    get_client, get_db, seed_all as mongo_seed,
    get_customers_with_embeddings, get_products_by_category,
    get_purchases_by_customer
)
from src.app.db.graph.neo4j_service import (
    get_driver, create_graph_schema,
    get_customers_who_bought_x_also_bought_y,
    get_customer_purchase_history,
    get_products_by_category_recommendation,
    get_top_products_overall
)

---
## 2. PostgreSQL — Relational Data Modeling (3NF)

**Schema:** products, customers, purchases, user_ratings

**Design Decisions:**
- All tables in 3NF — no transitive dependencies
- Proper foreign keys with referential integrity
- Numeric types for prices/quantities, TIMESTAMPTZ for dates, TEXT for descriptions
- Indexes on foreign keys and frequently queried columns
- CHECK constraints for data validation

In [ ]:
async def init_pg():
    conn = await get_connection()
    try:
        await execute_ddl(conn)
        counts = await seed_all(conn)
        return counts
    finally:
        await conn.close()

counts = await init_pg()
print('PostgreSQL seeded:', counts)

In [ ]:
conn = await get_connection()
try:
    # Query 1: Customers with their purchases
    print("=== Customers with Purchases ===")
    rows = await fetch_customers_with_purchases(conn, 5)
    for r in rows:
        print(f"  {r['first_name']} {r['last_name']} → {r.get('product_name', 'N/A')} (${r.get('total_amount', 0)})")

    # Query 2: Top products by sales
    print("\n=== Top Products ===")
    top = await fetch_top_products(conn, 5)
    for r in top:
        print(f"  {r['name']} — Sold: {r['total_sold']}, Avg Rating: {r['avg_rating']}")

    # Query 3: Recommendations (customers who bought X also bought Y)
    print("\n=== Product Recommendations for 'Ergonomic Office Chair' ===")
    recs = await fetch_product_recommendations(conn, 1, 5)
    for r in recs:
        print(f"  {r['name']} (bought together {r['times_bought_together']} times)")
finally:
    await conn.close()

---
## 3. MongoDB — Document Database Modeling

**Collections:** customers, products, purchases

**Design Decisions:**
- **Embedding:** Customer documents embed 10 most recent purchases and industry-matched products
- **Referencing:** Purchase documents reference customer_id and product_id (strings)
- **Flexible Schema:** Product documents have varying fields (e.g., Keyboard has switch_type, Monitor has resolution/ports)
- 3 collections total with at least 2 products having varying attribute fields

In [ ]:
client = get_client()
db = get_db(client)
counts = await mongo_seed(db)
print('MongoDB seeded:', counts)
client.close()

In [ ]:
client = get_client()
db = get_db(client)

# Query 1: Get customers with embedded documents
print("=== Customers with Embedded Purchases ===")
customers = await get_customers_with_embeddings(db, 3)
for c in customers:
    print(f"  {c['first_name']} {c['last_name']} ({c['industry']})")
    print(f"    Recent purchases: {len(c.get('recent_purchases', []))} items")
    print(f"    Matched products: {len(c.get('matched_products', []))} items")

# Query 2: Filter products by category
print("\n=== Electronics Products ===")
electronics = await get_products_by_category(db, 'Electronics')
for p in electronics:
    print(f"  {p['name']} — ${p['unit_price']}")

# Query 3: Purchases by customer (with references)
print("\n=== Purchases by Alice Johnson ===")
purchases = await get_purchases_by_customer(db, 'alice.johnson@email.com')
for p in purchases:
    print(f"  Purchase of product {p['product_id']} — ${p['total_amount']}")

client.close()

---
## 4. Neo4j — Graph Database Modeling

**Nodes:** Customer, Product, Category

**Relationships:**
- `PURCHASED` (Customer → Product) — with properties: quantity, amount, date
- `ALSO_BOUGHT` (Product → Product) — with strength property for collaborative filtering
- `BELONGS_TO` (Product → Category) — additional relationship type

**Design Decisions:**
- Category as additional node type for hierarchical product organization
- ALSO_BOUGHT strength scores enable weighted collaborative filtering
- BELONGS_TO enables category-based recommendation queries

In [ ]:
driver = get_driver()
async with driver:
    await create_graph_schema(driver)
print('Neo4j schema created')

In [ ]:
driver = get_driver()
async with driver:
    # Recommendation: Customers who bought X also bought Y
    print("=== 'Customers who bought Ergonomic Office Chair also bought...' ===")
    recs = await get_customers_who_bought_x_also_bought_y(driver, 'Ergonomic Office Chair', 5)
    for r in recs:
        print(f"  {r['product_name']} ({r['category']}) — score: {r['recommendation_score']}")

    # Customer purchase history
    print("\n=== Purchase History for Alice Johnson ===")
    history = await get_customer_purchase_history(driver, 'alice.johnson@email.com')
    for h in history:
        print(f"  {h['product_name']} — ${h['amount']} on {h['purchase_date']}")

    # Category-based recommendation
    print("\n=== Products in Electronics Category ===")
    cat_products = await get_products_by_category_recommendation(driver, 'Electronics')
    for p in cat_products:
        print(f"  {p['product_name']} — ${p['unit_price']}")

    # Top products overall
    print("\n=== Top Products (by purchase count) ===")
    top = await get_top_products_overall(driver, 5)
    for t in top:
        print(f"  {t['product_name']} — {t['purchase_count']} purchases, ${t['total_revenue']} revenue")

---
## 5. Architecture Summary

| Database | Type | Purpose | Key Features |
|---|---|---|---|
| PostgreSQL | Relational (3NF) | OLTP, transactions, integrity | Foreign keys, CHECK constraints, indexes |
| MongoDB | Document | Flexible schemas, embedded data | Embedding (recent purchases), referencing, varying product fields |
| Neo4j | Graph | Recommendations, relationships | PURCHASED, ALSO_BOUGHT, BELONGS_TO, Cypher traversal |

**API Endpoints:**
- `POST /api/oltp/init` — Initialize PostgreSQL
- `GET /api/oltp/customers-with-purchases` — Customers with purchase history
- `GET /api/oltp/top-products` — Top selling products
- `GET /api/oltp/recommendations/{product_id}` — Product recommendations
- `POST /api/nosql/init` — Initialize MongoDB
- `GET /api/nosql/customers` — Customers with embedded docs
- `GET /api/nosql/products/{category}` — Products by category
- `GET /api/nosql/purchases/{email}` — Purchases by customer email
- `POST /api/graph/init` — Initialize Neo4j
- `GET /api/graph/recommendations/{product_name}` — Also-bought recommendations
- `GET /api/graph/customer-history/{email}` — Customer purchase history
- `GET /api/graph/category-products/{category}` — Products in category
- `GET /api/graph/top-products` — Top products in graph